In [2]:
bk_zip_codes = ['11201', '11206', '11207', '11208', '11209', '11202', '11203', '11204', '11205', '11210', '11211', '11212', '11213', '11218', '11219', '11220', '11221', '11222', '11223', '11224', '11225', '11214', '11215', '11216', '11217', '11226', '11228', '11229', '11230', '11235', '11236', '11237', '11238', '11245', '11247', '11249', '11256', '11231', '11232', '11233', '11234', '11239', '11241', '11242', '11243', '11251', '11252']
len(bk_zip_codes)

47

In [3]:
import os
from dotenv import load_dotenv
load_dotenv()
import requests
census_api_key = os.environ.get("CENSUS_API_KEY")
import pandas as pd

In [4]:



zcta_str = ",".join(bk_zip_codes)
# income breakdown from 2023 group
group = "S0101"

url = f"https://api.census.gov/data/2023/acs/acs5/subject?get=NAME,group(S1901)&for=zip%20code%20tabulation%20area:{zcta_str}&key={census_api_key}"


response = requests.get(url)

In [5]:
# Checking if the request was successful
data = None
if response.status_code == 200:
    data = response.json()
else:
    # Handling errors
    print(f"Error: {response.status_code}, {response.text}")

if data:
    data = pd.DataFrame(data)

data.columns = data.iloc[0]
data = data[1:]
# has lots of metrics we don't need
#also isnt human readable yet
data


,NAME,GEO_ID,NAME,S1901_C01_001E,S1901_C01_001EA,S1901_C01_001M,S1901_C01_001MA,S1901_C01_002E,S1901_C01_002EA,S1901_C01_002M,...,S1901_C04_014MA,S1901_C04_015E,S1901_C04_015EA,S1901_C04_015M,S1901_C04_015MA,S1901_C04_016E,S1901_C04_016EA,S1901_C04_016M,S1901_C04_016MA,zip code tabulation area
1,ZCTA5 11201,860Z200US11201,ZCTA5 11201,33141,None,1000,None,5.4,None,1.0,...,(X),-888888888,(X),-888888888,(X),28.9,None,-888888888.0,(X),11201
2,ZCTA5 11203,860Z200US11203,ZCTA5 11203,30149,None,820,None,7.3,None,1.1,...,(X),-888888888,(X),-888888888,(X),39.7,None,-888888888.0,(X),11203
3,ZCTA5 11204,860Z200US11204,ZCTA5 11204,25306,None,1091,None,4.8,None,0.9,...,(X),-888888888,(X),-888888888,(X),49.2,None,-888888888.0,(X),11204
4,ZCTA5 11205,860Z200US11205,ZCTA5 11205,17272,None,769,None,7.3,None,1.9,...,(X),-888888888,(X),-888888888,(X),34.5,None,-888888888.0,(X),11205
5,ZCTA5 11206,860Z200US11206,ZCTA5 11206,33696,None,1123,None,10.6,None,2.0,...,(X),-888888888,(X),-888888888,(X),36.2,None,-888888888.0,(X),11206
6,ZCTA5 11207,860Z200US11207,ZCTA5 11207,35309,None,1118,None,13.7,None,2.0,...,(X),-888888888,(X),-888888888,(X),51.1,None,-888888888.0,(X),11207
7,ZCTA5 11208,860Z200US11208,ZCTA5 11208,34486,None,1141,None,10.6,None,1.6,...,(X),-888888888,(X),-888888888,(X),72.8,None,-888888888.0,(X),11208
8,ZCTA5 11209,860Z200US11209,ZCTA5 11209,30376,None,711,None,4.7,None,0.7,...,(X),-888888888,(X),-888888888,(X),36.6,None,-888888888.0,(X),11209
9,ZCTA5 11210,860Z200US11210,ZCTA5 11210,21440,None,905,None,5.4,None,1.2,...,(X),-888888888,(X),-888888888,(X),40.4,None,-888888888.0,(X),11210
10,ZCTA5 11211,860Z200US11211,ZCTA5 11211,27497,None,1113,None,5.6,None,1.2,...,(X),-888888888,(X),-888888888,(X),37.1,None,-888888888.0,(X),11211


In [8]:


# Get variable definitions
vars_url = "https://api.census.gov/data/2023/acs/acs5/subject/variables.json"
vars_response = requests.get(vars_url, timeout=10).json()
variables = vars_response['variables']

# match labels for S1901 variables
s1901_labels = {
    var: info['label'] 
    for var, info in variables.items() 
    if var.startswith("S1901_C01_") and var.endswith("E")
}

s1901_labels = s1901_labels.items()
s1901_labels = pd.DataFrame(s1901_labels)
s1901_labels.rename(columns={0: 'code_label', 1: 'readable_label'}, inplace=True)
s1901_labels.columns

s1901_labels.to_csv("s1901_labels.csv", index=False)


In [10]:
col_names = list(s1901_labels['code_label'])
columns_to_select = col_names + ['GEO_ID', 'zip code tabulation area']
filtered_data = data[columns_to_select]
filtered_data = filtered_data.rename(columns={"zip code tabulation area": "ZIP"})

In [ ]:
import pandas as pd

def rename_columns_with_labels(data_df, labels_df):
    """
    Rename columns in data_df using human-readable labels from labels_df
    
    Args:
        data_df: DataFrame with code labels as column names
        labels_df: DataFrame with code_label and readable_label columns
    
    Returns:
        DataFrame with renamed columns
    """
    # Check if the expected columns exist in labels_df
    if 'code_label' not in labels_df.columns or 'readable_label' not in labels_df.columns:
        # Print the actual column names for debugging
        print(f"Available columns in labels dataframe: {labels_df.columns.tolist()}")
        raise ValueError("Labels dataframe must contain 'code_label' and 'readable_label' columns")
    
    # Clean the readable labels by replacing '!!' with a single space
    cleaned_labels = labels_df['readable_label'].str.replace('!!', ' ')
    
    # Create a dictionary mapping code labels to cleaned readable labels
    label_dict = dict(zip(labels_df['code_label'], cleaned_labels))
    
    # Create a copy of the original dataframe
    renamed_df = data_df.copy()
    
    # Rename columns that have corresponding readable labels
    columns_to_rename = {col: label_dict.get(col, col) for col in data_df.columns if col in label_dict}
    renamed_df = renamed_df.rename(columns=columns_to_rename)
    
    return renamed_df

renamed_df = rename_columns_with_labels(filtered_data, s1901_labels)

# Display the result
renamed_df.head()

,Estimate Households PERCENT ALLOCATED Nonfamily income in the past 12 months,Estimate Households PERCENT ALLOCATED Family income in the past 12 months,Estimate Households PERCENT ALLOCATED Household income in the past 12 months,Estimate Households Mean income (dollars),Estimate Households Median income (dollars),"Estimate Households Total $200,000 or more","Estimate Households Total $150,000 to $199,999","Estimate Households Total $100,000 to $149,999","Estimate Households Total $75,000 to $99,999","Estimate Households Total $50,000 to $74,999","Estimate Households Total $35,000 to $49,999","Estimate Households Total $25,000 to $34,999","Estimate Households Total $15,000 to $24,999","Estimate Households Total $10,000 to $14,999","Estimate Households Total Less than $10,000",Estimate Households Total,GEO_ID,ZIP
1,-888888888,-888888888,30.0,243020,169285,42.7,11.9,14.5,5.6,6.4,4.7,3.4,3.3,2.2,5.4,33141,860Z200US11201,11201
2,-888888888,-888888888,58.1,88652,68028,9.3,7.3,15.4,13.8,14.7,11.6,8.3,6.7,5.6,7.3,30149,860Z200US11203,11203
3,-888888888,-888888888,51.7,88033,67588,6.3,9.2,15.5,15.1,18.1,9.7,8.2,8.0,5.1,4.8,25306,860Z200US11204,11204
4,-888888888,-888888888,45.8,142490,86753,22.4,10.1,14.4,8.6,13.6,6.6,5.5,7.0,4.6,7.3,17272,860Z200US11205,11205
5,-888888888,-888888888,44.4,88480,57280,10.2,7.5,14.9,8.8,12.4,10.4,7.9,10.2,7.2,10.6,33696,860Z200US11206,11206


In [ ]:
# BK DATA FOR 2023 INCOME BREAKDOWN BY ZIPCODE
bk_income_data_by_zip = renamed_df

bk_income_data_by_zip.to_csv("bk_income_data_by_zip.csv", index=False)